# Reflection & Self-Critique — Agents That Improve Their Own Work

## What This Notebook Teaches You

Humans rarely get things right on the first try. A writer drafts, re-reads, critiques, and revises. A programmer writes code, tests it, finds bugs, and fixes them. **Reflection agents** bring this same generate-critique-revise loop to LLMs.

**Research foundation**:
- Shinn et al. 2023, *"Reflexion: Language Agents with Verbal Reinforcement Learning"* — agents that reflect on failures and store insights in memory
- Madaan et al. 2023, *"Self-Refine: Iterative Refinement with Self-Feedback"* — showed that LLMs can iteratively improve their own outputs through self-critique

By the end of this notebook, you will:

1. **Understand the generate-critique-revise loop** and why it outperforms single-shot generation
2. **Build evaluation rubrics** with weighted, multi-dimensional scoring criteria
3. **Implement a full ReflectionAgent** with generation, critique, revision, and convergence detection
4. **Apply reflection to 3 domains** — writing, code generation, and analysis
5. **Visualize quality improvement** across iterations
6. **Know when reflection helps and when it hurts**

---

> **Prerequisites**: Notebooks 01–03 (agent fundamentals, tool use, ReAct).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~50–65 minutes.

## Part 0: Environment Setup

Same Qwen3-8B setup. The model will play three roles: generator, critic, and reviser — all controlled by different system prompts.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Reflection?

### 1.1 — The Problem with Single-Shot Generation

LLMs generate text left-to-right, committing to each token as they go. This means:
- **No revision** — early mistakes propagate through the entire output
- **No self-awareness** — the model cannot "re-read" what it wrote and spot errors
- **Decoding artifacts** — repetition, drift, inconsistency accumulate

### 1.2 — The Human Analogy

Think of the writer-editor relationship:

```
Writer (Generate)          Editor (Critique)           Writer (Revise)
     │                          │                          │
     │  "Here's my draft"       │                          │
     ├─────────────────────────►│                          │
     │                          │  "Paragraph 2 is vague,  │
     │                          │   conclusion is weak"     │
     │                          ├─────────────────────────►│
     │                          │                          │  "Here's v2 with
     │                          │                          │   fixes applied"
     │                          │◄─────────────────────────┤
     │                          │  "Much better, but the   │
     │                          │   opening could hook      │
     │                          │   the reader more"        │
     │                          ├─────────────────────────►│
     │                          │                          │  Final version v3
```

### 1.3 — The Generate-Critique-Revise Loop

```
                    ┌──────────────┐
                    │   GENERATE   │◄──── Task prompt
                    └──────┬───────┘
                           │ draft
                           ▼
                    ┌──────────────┐
              ┌────►│   CRITIQUE   │
              │     └──────┬───────┘
              │            │ scores + feedback
              │            ▼
              │     ┌──────────────┐
              │     │   REVISE     │
              │     └──────┬───────┘
              │            │ improved draft
              │            ▼
              │     ┌──────────────┐
              └─────│  CONVERGED?  │────► YES → Final output
                    └──────────────┘
```

Each iteration costs more tokens, but the output quality improves — up to a point.

## 2. Building Evaluation Rubrics

Good critique needs **structure**. Vague feedback ("make it better") doesn't help. We build rubrics with named dimensions, descriptions, and scoring scales.

In [ ]:
@dataclass
class RubricDimension:
    """A single dimension of evaluation."""
    name: str
    description: str
    weight: float = 1.0

@dataclass 
class Rubric:
    """A complete evaluation rubric with multiple dimensions."""
    name: str
    dimensions: List[RubricDimension]
    
    def to_prompt(self) -> str:
        """Convert rubric to a prompt-friendly format."""
        lines = [f"Evaluation Rubric: {self.name}\n"]
        lines.append("Score each dimension from 1 (poor) to 5 (excellent):\n")
        for d in self.dimensions:
            lines.append(f"- **{d.name}** (weight: {d.weight}): {d.description}")
            lines.append(f"  1=Very Poor  2=Poor  3=Adequate  4=Good  5=Excellent")
        lines.append("\nFor EACH dimension, provide:")
        lines.append("1. Score (1-5)")
        lines.append("2. Specific feedback explaining the score")
        lines.append("3. Concrete suggestion for improvement")
        lines.append("\nFormat your response as JSON:")
        lines.append('{"scores": {"dimension_name": {"score": N, "feedback": "...", "suggestion": "..."}}, "overall_feedback": "..."}')
        return "\n".join(lines)
    
    def total_weight(self) -> float:
        return sum(d.weight for d in self.dimensions)

# Pre-built rubrics for different domains
WRITING_RUBRIC = Rubric(
    name="Writing Quality",
    dimensions=[
        RubricDimension("clarity", "Is the writing clear and easy to understand?", 1.5),
        RubricDimension("accuracy", "Are all facts and claims correct?", 2.0),
        RubricDimension("completeness", "Does it cover all important aspects of the topic?", 1.5),
        RubricDimension("engagement", "Is the writing engaging and interesting to read?", 1.0),
        RubricDimension("structure", "Is the text well-organized with logical flow?", 1.0),
    ]
)

CODE_RUBRIC = Rubric(
    name="Code Quality",
    dimensions=[
        RubricDimension("correctness", "Does the code produce correct output for all cases?", 2.5),
        RubricDimension("efficiency", "Is the algorithm efficient in time and space?", 1.5),
        RubricDimension("readability", "Is the code clean, well-named, and easy to follow?", 1.0),
        RubricDimension("edge_cases", "Does the code handle edge cases and errors?", 1.5),
        RubricDimension("style", "Does the code follow Python best practices?", 0.5),
    ]
)

ANALYSIS_RUBRIC = Rubric(
    name="Analysis Quality",
    dimensions=[
        RubricDimension("balance", "Does the analysis present multiple perspectives fairly?", 2.0),
        RubricDimension("evidence", "Are claims supported with reasoning or evidence?", 2.0),
        RubricDimension("depth", "Does the analysis go beyond surface-level observations?", 1.5),
        RubricDimension("clarity", "Is the analysis clearly written and well-structured?", 1.0),
        RubricDimension("actionability", "Does the analysis lead to useful conclusions?", 1.0),
    ]
)

print("Writing rubric prompt preview:")
print(WRITING_RUBRIC.to_prompt()[:500] + "...")

## 3. The Reflection Agent

### 3.1 — Parsing Critique Results

In [ ]:
@dataclass
class CritiqueResult:
    """Parsed result from a critique."""
    dimension_scores: Dict[str, dict]  # {name: {score, feedback, suggestion}}
    overall_feedback: str
    weighted_score: float = 0.0
    
    @classmethod
    def parse(cls, raw_output: str, rubric: Rubric) -> 'CritiqueResult':
        """Parse LLM critique output into structured result."""
        json_match = re.search(r'\{[\s\S]*\}', raw_output)
        if json_match:
            try:
                data = json.loads(json_match.group())
                scores = data.get("scores", {})
                overall = data.get("overall_feedback", "No overall feedback.")
                
                result = cls(dimension_scores=scores, overall_feedback=overall)
                result.weighted_score = result._compute_weighted_score(rubric)
                return result
            except json.JSONDecodeError:
                pass
        
        # Fallback: create default scores if parsing fails
        default_scores = {}
        for d in rubric.dimensions:
            default_scores[d.name] = {
                "score": 3, "feedback": "Could not parse critique.", "suggestion": "N/A"
            }
        return cls(
            dimension_scores=default_scores,
            overall_feedback=raw_output[:300],
            weighted_score=3.0
        )
    
    def _compute_weighted_score(self, rubric: Rubric) -> float:
        """Compute weighted average score."""
        total_weight = 0
        weighted_sum = 0
        for d in rubric.dimensions:
            if d.name in self.dimension_scores:
                score = self.dimension_scores[d.name].get("score", 3)
                weighted_sum += score * d.weight
                total_weight += d.weight
        return weighted_sum / total_weight if total_weight > 0 else 3.0
    
    def summary(self) -> str:
        lines = [f"Weighted Score: {self.weighted_score:.2f}/5.00"]
        for name, data in self.dimension_scores.items():
            score = data.get("score", "?")
            feedback = data.get("feedback", "N/A")[:80]
            lines.append(f"  {name}: {score}/5 — {feedback}")
        lines.append(f"Overall: {self.overall_feedback[:150]}")
        return "\n".join(lines)

print("✓ CritiqueResult class defined")

### 3.2 — The Full Reflection Agent

In [ ]:
class ReflectionAgent:
    """Agent that generates, critiques, and revises its own output."""
    
    def __init__(self, rubric: Rubric, max_iterations: int = 3):
        self.rubric = rubric
        self.max_iterations = max_iterations
        self.history = []  # List of (draft, critique) tuples
    
    def generate(self, task: str) -> str:
        """Generate initial draft."""
        messages = [
            {"role": "system", "content": "You are a skilled writer/coder. Produce high-quality output for the given task. Be thorough and precise."},
            {"role": "user", "content": task}
        ]
        return generate(messages, max_new_tokens=700, temperature=0.7)
    
    def critique(self, task: str, output: str) -> CritiqueResult:
        """Critique the output using the rubric."""
        rubric_prompt = self.rubric.to_prompt()
        messages = [
            {"role": "system", "content": f"You are a strict but fair critic. Evaluate the given output using the rubric below.\n\n{rubric_prompt}"},
            {"role": "user", "content": f"TASK: {task}\n\nOUTPUT TO EVALUATE:\n{output}"}
        ]
        raw = generate(messages, max_new_tokens=600, temperature=0.3, do_sample=True)
        return CritiqueResult.parse(raw, self.rubric)
    
    def revise(self, task: str, output: str, critique: CritiqueResult) -> str:
        """Revise the output based on critique feedback."""
        feedback_text = "\n".join(
            f"- {name}: Score {data.get('score', '?')}/5. {data.get('suggestion', 'N/A')}"
            for name, data in critique.dimension_scores.items()
        )
        messages = [
            {"role": "system", "content": "You are revising your previous work based on critique feedback. Keep what was good, fix what was criticized. Produce the COMPLETE revised version, not just changes."},
            {"role": "user", "content": f"TASK: {task}\n\nCURRENT VERSION:\n{output}\n\nCRITIQUE FEEDBACK:\n{feedback_text}\n\nOverall: {critique.overall_feedback}\n\nProduce the complete revised version:"}
        ]
        return generate(messages, max_new_tokens=700, temperature=0.7)
    
    def run(self, task: str, verbose: bool = True) -> Tuple[str, List[dict]]:
        """Run the full generate-critique-revise loop."""
        if verbose:
            print(f"{'='*60}")
            print(f"TASK: {task}")
            print(f"Rubric: {self.rubric.name} ({len(self.rubric.dimensions)} dimensions)")
            print(f"Max iterations: {self.max_iterations}")
            print(f"{'='*60}\n")
        
        self.history = []
        
        # Initial generation
        if verbose:
            print("[Iteration 0: GENERATE]")
        current_output = self.generate(task)
        if verbose:
            print(f"Draft (first 300 chars):\n{current_output[:300]}...\n")
        
        for iteration in range(self.max_iterations):
            # Critique
            if verbose:
                print(f"[Iteration {iteration + 1}: CRITIQUE]")
            critique_result = self.critique(task, current_output)
            if verbose:
                print(critique_result.summary())
                print()
            
            self.history.append({
                "iteration": iteration,
                "output": current_output,
                "critique": critique_result,
                "score": critique_result.weighted_score
            })
            
            # Check convergence
            if critique_result.weighted_score >= 4.5:
                if verbose:
                    print(f"✓ Score {critique_result.weighted_score:.2f} >= 4.5 — converged!")
                break
            
            if len(self.history) >= 2:
                prev_score = self.history[-2]["score"]
                improvement = critique_result.weighted_score - prev_score
                if improvement < 0.1:
                    if verbose:
                        print(f"△ Improvement {improvement:.2f} < 0.1 — diminishing returns, stopping.")
                    break
            
            # Revise
            if verbose:
                print(f"[Iteration {iteration + 1}: REVISE]")
            current_output = self.revise(task, current_output, critique_result)
            if verbose:
                print(f"Revised (first 300 chars):\n{current_output[:300]}...\n")
        
        return current_output, self.history
    
    def score_trajectory(self) -> List[float]:
        """Return the scores across iterations."""
        return [h["score"] for h in self.history]

print("✓ ReflectionAgent defined")

## 4. Experiment 1: Text Writing

### "Explain Photosynthesis"

We use the writing rubric to evaluate clarity, accuracy, completeness, engagement, and structure across 3 iterations.

In [ ]:
writing_agent = ReflectionAgent(rubric=WRITING_RUBRIC, max_iterations=3)
final_text, text_history = writing_agent.run(
    "Write a clear, engaging explanation of photosynthesis for a college biology student. "
    "Cover the light-dependent and light-independent reactions, the role of chlorophyll, "
    "and why photosynthesis matters for life on Earth."
)

print(f"\n{'='*60}")
print("FINAL OUTPUT:")
print(f"{'='*60}")
print(final_text)

In [ ]:
# Show score trajectory
scores = writing_agent.score_trajectory()
print("Score Trajectory (Writing Task):")
print(f"{'Iteration':<12} {'Score':<8} {'Bar'}")
print("-" * 50)
for i, score in enumerate(scores):
    bar = "█" * int(score * 6) + "░" * (30 - int(score * 6))
    print(f"  {i:<10} {score:<8.2f} {bar}")
print(f"\nImprovement: {scores[0]:.2f} → {scores[-1]:.2f} ({scores[-1] - scores[0]:+.2f})")

## 5. Experiment 2: Code Generation

### "Write a Prime Sieve Function"

The code rubric emphasizes correctness, efficiency, and edge case handling. The critic can catch bugs that the generator missed.

In [ ]:
code_agent = ReflectionAgent(rubric=CODE_RUBRIC, max_iterations=3)
final_code, code_history = code_agent.run(
    "Write a Python function 'sieve_of_eratosthenes(n)' that returns a list of all prime numbers "
    "up to n (inclusive). Handle edge cases (n < 2, n = 2, large n). Include a docstring. "
    "The function should be efficient and follow Python best practices."
)

print(f"\n{'='*60}")
print("FINAL CODE:")
print(f"{'='*60}")
print(final_code)

In [ ]:
# Show code score trajectory
code_scores = code_agent.score_trajectory()
print("Score Trajectory (Code Task):")
print(f"{'Iteration':<12} {'Score':<8} {'Dimensions'}")
print("-" * 70)
for i, entry in enumerate(code_history):
    dims = entry["critique"].dimension_scores
    dim_str = "  ".join(
        f"{k}:{v.get('score','?')}" for k, v in dims.items()
    )
    print(f"  {i:<10} {entry['score']:<8.2f} {dim_str}")

## 6. Experiment 3: Balanced Analysis

### "Pros and Cons of Remote Work"

The analysis rubric checks balance, evidence, and depth. The critic should push the agent toward more nuanced, well-supported arguments.

In [ ]:
analysis_agent = ReflectionAgent(rubric=ANALYSIS_RUBRIC, max_iterations=3)
final_analysis, analysis_history = analysis_agent.run(
    "Write a balanced analysis of the pros and cons of remote work for both employees and employers. "
    "Consider productivity, mental health, collaboration, cost, and work-life balance. "
    "Support claims with reasoning and conclude with a nuanced recommendation."
)

print(f"\n{'='*60}")
print("FINAL ANALYSIS:")
print(f"{'='*60}")
print(final_analysis)

## 7. Quality Curves — Visualizing Improvement

Let's visualize the score trajectories across all three experiments using text-based charts.

In [ ]:
def text_chart(title: str, trajectories: Dict[str, List[float]], width: int = 40):
    """Render a text-based quality chart."""
    print(f"\n{title}")
    print("=" * (width + 25))
    print(f"{'Label':<15} {'Score Plot (1-5 scale)':<{width+10}}")
    print("-" * (width + 25))
    
    for label, scores in trajectories.items():
        for i, score in enumerate(scores):
            tag = f"{label}[{i}]"
            filled = int((score / 5.0) * width)
            bar = "█" * filled + "░" * (width - filled)
            marker = f" {score:.2f}"
            print(f"  {tag:<13} |{bar}|{marker}")
        print()  # blank line between trajectories
    
    print("-" * (width + 25))
    
    # Summary
    print("\nSummary:")
    for label, scores in trajectories.items():
        delta = scores[-1] - scores[0] if len(scores) > 1 else 0
        print(f"  {label}: {scores[0]:.2f} → {scores[-1]:.2f} (Δ{delta:+.2f}, {len(scores)} iterations)")

all_trajectories = {
    "Writing": writing_agent.score_trajectory(),
    "Code": code_agent.score_trajectory(),
    "Analysis": analysis_agent.score_trajectory(),
}

text_chart("Quality Improvement Across Domains", all_trajectories)

## 8. Convergence Detection

Reflection isn't free — each iteration costs tokens. We need strategies to stop when improvement is no longer worth the cost.

### Convergence Strategies

| Strategy | Stop When | Best For |
|----------|----------|----------|
| **Score threshold** | Score ≥ 4.5 | Tasks with clear quality targets |
| **Improvement delta** | Δscore < 0.1 | Detecting diminishing returns |
| **Max iterations** | i ≥ max_iter | Budget constraints |
| **Score decline** | score[i] < score[i-1] | Preventing over-editing |

Our ReflectionAgent already implements threshold + delta convergence. Let's test the score decline strategy.

In [ ]:
class ConvergenceDetector:
    """Configurable convergence detection for reflection loops."""
    
    def __init__(
        self,
        score_threshold: float = 4.5,
        min_improvement: float = 0.1,
        max_iterations: int = 5,
        detect_decline: bool = False
    ):
        self.score_threshold = score_threshold
        self.min_improvement = min_improvement
        self.max_iterations = max_iterations
        self.detect_decline = detect_decline
    
    def should_stop(self, scores: List[float], iteration: int) -> Tuple[bool, str]:
        """Check if we should stop iterating. Returns (stop, reason)."""
        if iteration >= self.max_iterations:
            return True, f"Max iterations ({self.max_iterations}) reached"
        
        if scores and scores[-1] >= self.score_threshold:
            return True, f"Score {scores[-1]:.2f} >= threshold {self.score_threshold}"
        
        if len(scores) >= 2:
            delta = scores[-1] - scores[-2]
            if delta < self.min_improvement:
                return True, f"Improvement {delta:.3f} < minimum {self.min_improvement}"
            if self.detect_decline and delta < 0:
                return True, f"Score declined: {scores[-2]:.2f} → {scores[-1]:.2f}"
        
        return False, "Continue"

# Demonstrate convergence scenarios
detector = ConvergenceDetector(detect_decline=True)

test_scenarios = [
    ("Steady improvement", [2.5, 3.2, 3.8, 4.3, 4.6]),
    ("Diminishing returns", [2.8, 3.5, 3.6, 3.62]),
    ("Score decline", [3.0, 3.8, 3.5]),
    ("Fast convergence", [3.5, 4.8]),
]

print("Convergence Detection Tests:")
print("-" * 60)
for name, scores in test_scenarios:
    for i in range(len(scores)):
        stop, reason = detector.should_stop(scores[:i+1], i)
        if stop:
            print(f"  {name}: STOP at iteration {i} — {reason}")
            break
    else:
        print(f"  {name}: Completed all {len(scores)} iterations")

## 9. When Reflection Hurts

Reflection is not always beneficial. Here are cases where it can **degrade** output quality:

### 9.1 — Simple Tasks (Over-Thinking)

For straightforward tasks like "What is 2+2?" or "Translate 'hello' to French", reflection adds cost without benefit. The first draft is usually correct.

### 9.2 — Over-Editing

Multiple revision cycles can make text **worse** by:
- **Hedging** — adding too many qualifiers ("it could possibly be argued that...")
- **Bloating** — making text longer without adding substance
- **Loss of voice** — each revision smooths out distinctive style
- **Circular edits** — changing X to Y in iteration 2, then back to X in iteration 3

### 9.3 — Critique Hallucination

The critic may invent problems that don't exist, leading the reviser to "fix" things that were already correct. This is especially common with:
- Correct code that the critic misreads
- Factually accurate statements the critic second-guesses
- Stylistic choices flagged as "errors"

In [ ]:
# Demonstrate over-thinking on a simple task
print("Experiment: Reflection on a SIMPLE task (potential over-thinking)")
print("=" * 60)

simple_agent = ReflectionAgent(rubric=WRITING_RUBRIC, max_iterations=3)
simple_result, simple_history = simple_agent.run(
    "Write a one-sentence definition of gravity."
)

print(f"\nScore trajectory: {simple_agent.score_trajectory()}")
print(f"\nIteration 0 output (original):")
print(f"  {simple_history[0]['output'][:200]}")
print(f"\nFinal output:")
print(f"  {simple_result[:200]}")
print("\n→ For simple tasks, the first draft is often best. Reflection adds cost without clear benefit.")

## 10. Reflection vs No-Reflection Comparison

Let's run 4 tasks with and without reflection, comparing quality.

In [ ]:
# Define comparison tasks
comparison_tasks = [
    ("Simple fact", "Define the term 'machine learning' in one clear paragraph."),
    ("Explanation", "Explain how a neural network learns, covering forward pass, loss, and backpropagation."),
    ("Code", "Write a Python function to check if a string is a valid palindrome, ignoring spaces and punctuation."),
    ("Analysis", "Analyze the environmental impact of electric vehicles vs gasoline vehicles."),
]

comparison_results = []

for task_name, task_prompt in comparison_tasks:
    print(f"\n{'='*60}")
    print(f"Task: {task_name}")
    print(f"{'='*60}")
    
    # No reflection (single shot)
    print("\n--- Single-Shot (no reflection) ---")
    single_msg = [
        {"role": "system", "content": "Produce high-quality output for the given task."},
        {"role": "user", "content": task_prompt}
    ]
    single_start = time.time()
    single_output = generate(single_msg, max_new_tokens=500, temperature=0.7)
    single_time = time.time() - single_start
    
    # Score it with writing rubric
    rubric = CODE_RUBRIC if "code" in task_name.lower() else WRITING_RUBRIC
    scorer = ReflectionAgent(rubric=rubric, max_iterations=0)
    single_critique = scorer.critique(task_prompt, single_output)
    print(f"  Score: {single_critique.weighted_score:.2f}, Time: {single_time:.1f}s")
    
    # With reflection
    print("\n--- With Reflection (3 iterations) ---")
    reflect_agent = ReflectionAgent(rubric=rubric, max_iterations=3)
    reflect_start = time.time()
    reflect_output, reflect_history = reflect_agent.run(task_prompt, verbose=False)
    reflect_time = time.time() - reflect_start
    
    # Get final score
    final_critique = scorer.critique(task_prompt, reflect_output)
    reflect_scores = reflect_agent.score_trajectory()
    print(f"  Score: {final_critique.weighted_score:.2f}, Time: {reflect_time:.1f}s")
    print(f"  Trajectory: {' → '.join(f'{s:.2f}' for s in reflect_scores)}")
    
    comparison_results.append({
        "task": task_name,
        "single_score": single_critique.weighted_score,
        "single_time": single_time,
        "reflect_score": final_critique.weighted_score,
        "reflect_time": reflect_time,
        "iterations": len(reflect_scores),
    })

# Print comparison table
print(f"\n\n{'='*80}")
print("COMPARISON: Single-Shot vs Reflection")
print(f"{'='*80}")
print(f"{'Task':<20} {'Single Score':>13} {'Reflect Score':>14} {'Delta':>7} {'Single Time':>12} {'Reflect Time':>13}")
print("-" * 79)
for r in comparison_results:
    delta = r["reflect_score"] - r["single_score"]
    print(f"{r['task']:<20} {r['single_score']:>13.2f} {r['reflect_score']:>14.2f} {delta:>+7.2f} {r['single_time']:>11.1f}s {r['reflect_time']:>12.1f}s")
print("-" * 79)

avg_delta = sum(r["reflect_score"] - r["single_score"] for r in comparison_results) / len(comparison_results)
print(f"\nAverage improvement from reflection: {avg_delta:+.2f} points")

## 11. Key Takeaways

1. **The generate-critique-revise loop is simple but powerful** — a single LLM can play all three roles (generator, critic, reviser) with different system prompts.

2. **Structured rubrics produce structured feedback** — dimensions with scores give the reviser specific, actionable guidance rather than vague "make it better" instructions.

3. **Reflection typically improves output by 0.5–1.5 points** (on a 5-point scale) across diverse tasks, with the biggest gains on the first revision.

4. **Convergence detection saves tokens** — stop when scores plateau, decline, or exceed a threshold. Most improvement happens in iterations 1-2.

5. **Reflection hurts on simple tasks** — the overhead isn't justified, and over-editing can degrade quality through hedging, bloating, or circular edits.

6. **The critic can hallucinate problems** — false negatives in critique lead to unnecessary changes. Using structured rubrics mitigates but doesn't eliminate this.

7. **This pattern generalizes** — reflection works for writing, code, analysis, design, and any domain where quality can be evaluated. The rubric is the key customization point.

---

**Next notebook**: We'll explore *Tree of Thought* — branching into multiple reasoning paths and evaluating which is best (Notebook 08).